In [3]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [4]:
import pandas as pd
data=pd.read_csv("./data/demo.csv",encoding="gbk").sample(frac=1)
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
845,846,0,3,"Abbing, Mr. Anthony",male,42.0,0,0,C.A. 5547,7.5500,NaN,S
184,185,1,3,"Kink-Heilmann, Miss. Luise Gretchen",female,4.0,0,2,315153,22.0250,NaN,S
85,86,1,3,"Backstrom, Mrs. Karl Alfred (Maria Mathilda Gu...",female,33.0,3,0,3101278,15.8500,NaN,S
230,231,1,1,"Harris, Mrs. Henry Birkhardt (Irene Wallach)",female,35.0,1,0,36973,83.4750,C83,S
192,193,1,3,"Andersen-Jensen, Miss. Carla Christine Nielsine",female,19.0,1,0,350046,7.8542,NaN,S


In [5]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### 简单跑一个模型

In [11]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.classification import *

In [12]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
  .pipe(LabelEncoding(cols=["Name"]))\
  .pipe(LGBMClassification(y=y_train,native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},prefix="lgbm"))

x_test_new=table.fit(x_train).transform(x_test)
x_test_new.head(5)

,lgbm_0,lgbm_1
728,0.817892,0.182108
808,0.795626,0.204374
73,0.760301,0.239699
581,0.010906,0.989094
4,0.632412,0.367588


### 挂载

In [13]:
from easymlops.table.storage import KafkaStorage

In [14]:
table[-1].set_branch_pipe(KafkaStorage(bootstrap_servers="192.168.244.133:9092",topic_name="label_encoding",cols=['PassengerId','Pclass','Name','Sex','Age','SibSp','Parch',
                                                                             'Ticket','Fare','Cabin','Embarked','Pclass_3','Pclass_1','Pclass_2','Sex_male','Sex_female']))

### 模拟线上数据流

In [18]:
i=0
for record in tqdm(x_test[:5].to_dict("record")):
    table.transform_single(record,storage_base_dict={"key":i})
    i+=1

100%|███████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 192.19it/s]


### 模拟消费

In [1]:
from kafka import KafkaConsumer
consumer = KafkaConsumer('label_encoding', bootstrap_servers="192.168.244.133:9092",value_deserializer=lambda m: json.loads(m.decode('ascii')))

In [ ]:
for message in consumer:
    print(message)